In [1]:
import os
import shutil
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import mrcfile

In [ ]:
def actin_polarity_construct_particle_identifier_adhesion(k, i, zaehler):
    # Determine zeros for tomogram index
    if k < 10:
        tzeros = '000'
    elif k < 100:
        tzeros = '00'
    elif k < 1000:
        tzeros = '0'
    else:
        tzeros = ''

    # Determine zeros for filament index
    if i < 10:
        pzeros = '000'
    elif i < 100:
        pzeros = '00'
    elif i < 1000:
        pzeros = '0'
    else:
        pzeros = ''

    # Determine zeros for particle number
    if zaehler < 10:
        nzeros = '0000'
    elif zaehler < 100:
        nzeros = '000'
    elif zaehler < 1000:
        nzeros = '00'
    elif zaehler < 10000:
        nzeros = '0'
    else:
        nzeros = ''

    particle_identifier = f"t_{tzeros}{k}_f_{pzeros}{i}_p_{nzeros}{zaehler}"
    return particle_identifier


def ParseParticleList(particles_list_name, indx=None, particles_list_mod_name=None):
    """
    Parses a particle list file. The particle list is now assumed to consist of repeating groups of 4 lines:
    1. Particle line (path to .mrc)
    2. Particle wedge line (e.g., 'singleaxiswedge 30 80')
    3. Real space mask line (e.g., 'allpass')
    4. Blank line

    Parameters
    ----------
    particles_list_name : str
        Name of the particle list file to parse.
    indx : list or None
        Indices of particles to keep. If None, all particles are kept.
    particles_list_mod_name : str or None
        Name of the file to write the modified particle list to. If None, no output file is created.

    Returns
    -------
    particles_list_1_mod : list of str
        Filtered list of particle lines (MRC paths).
    particles_list_2_mod : list of str
        Filtered list of wedge lines.
    particles_list_3_mod : list of str
        Filtered list of mask lines (e.g. 'allpass').
    particles_list_4_mod : list of str
        Filtered list of the fourth lines (blank lines).
    """

    if indx is None:
        indx = []

    # Read file lines as-is, including empty lines
    with open(particles_list_name, 'r') as f:
        lines = f.read().splitlines()

    # Check that the number of lines is a multiple of 4
    if len(lines) % 4 != 0:
        raise ValueError("Particle list does not consist of groups of 4 lines.")

    # Partition lines into four separate lists
    particles_list_1 = lines[0::4]
    particles_list_2 = lines[1::4]
    particles_list_3 = lines[2::4]
    particles_list_4 = lines[3::4]

    # If indx is provided, filter the lists
    if len(indx) > 0:
        # If your indices are 1-based and you need to convert, uncomment the next line:
        # indx = [i-1 for i in indx]

        particles_list_1_mod = [particles_list_1[i] for i in indx if i < len(particles_list_1)]
        particles_list_2_mod = [particles_list_2[i] for i in indx if i < len(particles_list_2)]
        particles_list_3_mod = [particles_list_3[i] for i in indx if i < len(particles_list_3)]
        particles_list_4_mod = [particles_list_4[i] for i in indx if i < len(particles_list_4)]
    else:
        particles_list_1_mod = particles_list_1
        particles_list_2_mod = particles_list_2
        particles_list_3_mod = particles_list_3
        particles_list_4_mod = particles_list_4

    # Build modified particle list (4 lines per particle)
    particles_list_mod = []
    for p1, p2, p3, p4 in zip(particles_list_1_mod, particles_list_2_mod, particles_list_3_mod, particles_list_4_mod):
        particles_list_mod.extend([p1, p2, p3, p4])

    # If requested, write the modified particle list to a file
    if particles_list_mod_name is not None:
        with open(particles_list_mod_name, 'w') as fid:
            for line in particles_list_mod:
                fid.write(line + '\n')

    # Since we no longer have 'emfile' to remove, we skip that step.

    return particles_list_1_mod, particles_list_2_mod, particles_list_3_mod, particles_list_4_mod

def actin_polarity_parse_data_file_class2d_prealg_adhesion(relion_data_file):
    alg_struct = []

    with open(relion_data_file, 'r') as fid:
        for line in fid:
            line = line.strip()
            if not line:
                # End of file or empty line
                break

            # Split by whitespace
            values = line.split()

            # If the line does not have at least 14 fields, we assume end of relevant data
            if len(values) < 14:
                print("End of file reached or insufficient fields!")
                break

            # Create a dictionary for the current record
            record = {}
            record['data_line'] = values
            record['image_name'] = values[0]

            image_name_pre = record['image_name']
            # Find the '@' character
            at_pos = image_name_pre.find('@')
            if at_pos == -1:
                # If '@' not found, handle error or skip
                continue
            # Extract particle index from the image_name
            record['particle_indx'] = int(image_name_pre[:at_pos])

            record['micrograph_name'] = values[1]
            micrograph_name_pre = record['micrograph_name']

            # Adjust indexing from MATLAB (1-based) to Python (0-based)
            # MATLAB: micrograph_name_pre(91:94) means chars 91 through 94 (inclusive) 1-based.
            # Python: use micrograph_name_pre[90:94] because Python is 0-based.
            # Make sure this indexing is correct for your actual input.
            record['micrograph_indx'] = int(micrograph_name_pre[90:94])

            record['group_number'] = int(values[2])
            record['angle_rot'] = float(values[3])
            record['angle_tilt'] = float(values[4])
            record['angle_psi'] = float(values[5])
            record['origin_x'] = float(values[6])
            record['origin_y'] = float(values[7])
            record['class_indx'] = int(values[8])
            record['norm_corr'] = float(values[9])
            record['log_likeli_con'] = float(values[10])
            record['max_val_prob_dis'] = float(values[11])
            record['nr_sig_samples'] = float(values[12])

            alg_struct.append(record)

    return alg_struct


def actin_polarity_parse_data_file_class2d_finealg_adhesion(starfile_path):
    # Similar parsing function placeholder.
    return []

def actin_polarity_parse_data_file_class2d_finealg_adhesion_relion4(relion_data_file):
    """
    Parses a RELION data file for class2d fine alignment in relion4 format.

    The file is expected to contain columns in the order specified by the MATLAB code:
    
    1: _rlnImageName
    2: _rlnMicrographName
    3: _rlnAngleRotPrior
    4: _rlnAngleTiltPrior
    5: _rlnAnglePsiPrior
    6: _rlnOpticsGroup
    7: _rlnOriginXPriorAngst
    8: _rlnOriginYPriorAngst
    9: _rlnGroupNumber
    10: _rlnAngleRot
    11: _rlnAngleTilt
    12: _rlnAnglePsi
    13: _rlnOriginXAngst
    14: _rlnOriginYAngst
    15: _rlnClassNumber
    16: _rlnNormCorrection
    17: _rlnLogLikeliContribution
    18: _rlnMaxValueProbDistribution
    19: _rlnNrOfSignificantSamples
    
    Returns
    -------
    alg_struct : list of dicts
        Each dict corresponds to one particle and includes keys such as
        'image_name', 'micrograph_name', 'particle_indx', 'micrograph_indx', etc.
    """

    alg_struct = []
    with open(relion_data_file, 'r') as fid:
        for line in fid:
            line = line.strip()
            if not line:
                # Empty line: end of file or no more valid lines
                break
            
            values = line.split()
            # Check if we have at least 19 fields
            if len(values) < 19:
                print('End of file reached or insufficient fields!')
                break
            
            record = {}
            record['data_line'] = values

            record['image_name'] = values[0]
            image_name_pre = record['image_name']
            # Extract particle index (characters before '@')
            at_pos = image_name_pre.find('@')
            if at_pos == -1:
                # No '@' found; skip this line
                continue
            record['particle_indx'] = int(image_name_pre[:at_pos])

            record['micrograph_name'] = values[1]
            micrograph_name_pre = record['micrograph_name']
            # Extract micrograph index from last 8 chars (MATLAB: (end-7:end-4))
            record['micrograph_indx'] = int(micrograph_name_pre[-8:-4])

            record['AngleRotPrior'] = float(values[2])
            record['AngleTiltPrior'] = float(values[3])
            record['AnglePsiPrior'] = float(values[4])
            record['OpticsGroup'] = float(values[5])
            record['OriginXPriorAngst'] = float(values[6])
            record['OriginYPriorAngst'] = float(values[7])
            record['group_number'] = int(values[8])
            record['angle_rot'] = float(values[9])
            record['angle_tilt'] = float(values[10])
            record['angle_psi'] = float(values[11])
            record['OriginXAngst'] = float(values[12])
            record['OriginYAngst'] = float(values[13])
            record['class_indx'] = int(values[14])
            record['norm_corr'] = float(values[15])
            record['log_likeli_con'] = float(values[16])
            record['max_val_prob_dis'] = float(values[17])
            record['nr_sig_samples'] = float(values[18])

            alg_struct.append(record)

    return alg_struct

def actin_polarity_parse_data_file_class3d(starfile_path):
    # Another parsing function placeholder.
    return []

def actin_polarity_parse_data_file_class3d_2(relion_data_file):
    """
    Parses a RELION 3D classification data file (version 2).
    Expected columns based on the MATLAB code:
    
    1: _rlnImageName
    2: _rlnMicrographName
    3: _rlnAngleRotPrior
    4: _rlnAngleTiltPrior
    5: _rlnAnglePsiPrior
    6: _rlnGroupNumber
    7: _rlnAngleRot
    8: _rlnAngleTilt
    9: _rlnAnglePsi
    10: _rlnOriginX
    11: _rlnOriginY
    12: _rlnClassNumber
    13: _rlnNormCorrection
    14: _rlnRandomSubset
    15: _rlnLogLikeliContribution
    16: _rlnMaxValueProbDistribution
    17: _rlnNrOfSignificantSamples

    Returns
    -------
    alg_struct : list of dicts
        Each dict represents one particle's alignment parameters.
    """

    alg_struct = []
    with open(relion_data_file, 'r') as fid:
        for line in fid:
            line = line.strip()
            if not line:
                # End of file or empty line
                break

            values = line.split()

            # Check if we have at least 17 columns
            if len(values) < 17:
                print('End of file reached or insufficient fields!')
                break

            record = {}
            record['data_line'] = values
            record['image_name'] = values[0]
            image_name_pre = record['image_name']
            # Extract particle index from first 6 characters of image_name_pre
            # MATLAB: image_name_pre(1:6) -> Python: image_name_pre[0:6]
            record['particle_indx'] = int(image_name_pre[0:6])

            record['micrograph_name'] = values[1]
            micrograph_name_pre = record['micrograph_name']
            # MATLAB: micrograph_name_pre(21:21) -> a single character at position 21 (1-based)
            # Python: micrograph_name_pre[20] for zero-based indexing
            # Ensure that the micrograph_name is long enough. If not, adjust indexing.
            record['micrograph_indx'] = int(micrograph_name_pre[20])

            record['angle_rot'] = float(values[6])
            record['angle_tilt'] = float(values[7])
            record['angle_psi'] = float(values[8])
            record['origin_x'] = float(values[9])
            record['origin_y'] = float(values[10])
            record['class_indx'] = int(values[11])

            # If needed, you can also parse other columns (e.g., norm_corr, random_subset, etc.)
            # as in the previous examples. The MATLAB code doesn't store them here, so they are optional.

            alg_struct.append(record)

    return alg_struct

In [ ]:
import os
import numpy as np
import re

def symlink_particle(p, f_list, particlename, particle_dir, P_BasePath):
    fname = particlename[p]
    # extract the integer index from the filename
    m = re.search(r'-(\d+)\.mrc$', fname)
    if not m:
        print(f"[{p}] no index in '{fname}', skipping")
        return
    particleindx = int(m.group(1))

    # now grab the correlated 4th-column value
    f_val = int(f_list[particleindx - 1, 3])

    old_name = os.path.join(particle_dir, fname)
    new_ident = actin_polarity_construct_particle_identifier_adhesion(
        1,
        f_val,
        particleindx
    )
    new_name = os.path.join(P_BasePath, f"tau_particle_{new_ident}.mrc")

    try:
        os.symlink(old_name, new_name)
        print(f"[{p}] linked → {new_name}")
    except FileExistsError:
        print(f"[{p}] already exists: {new_name}")
    except OSError as e:
        print(f"[{p}] error linking '{old_name}': {e}")

    return new_name

# Load f_list and prepare directories and file lists as before
BasePath = '/mnt/storage/data/users/wen-lu/msa_1b_mouse/20250710_lift_out/Position_10_2/'
f_list_path = os.path.join(
    BasePath,'warp/Position_10_2/processing/reconstruction',
    'Position_10_2_ali_968Apx_clean.coords'
)
f_list = np.loadtxt(f_list_path)

particle_dir = os.path.join(BasePath,'ts-aligned/imod_2D/imod_subtomo_extract/particles_rec/')
particlename = [f for f in os.listdir(particle_dir) 
                if not f.startswith('.') 
                and 'ctf' not in f 
                and 'average' not in f 
                and not f.endswith('~')]
particlename.sort()

P_BasePath = os.path.join(BasePath,'ts-aligned/imod_2D/imod_subtomo_extract/particles_rec_rename/')
os.makedirs(P_BasePath, exist_ok=True)

# Process each file sequentially
for p in range(f_list.shape[0]):
    symlink_particle(p, f_list, particlename, particle_dir, P_BasePath)
    print(p)



In [ ]:
import os
PasePath = '/mnt/storage/data/users/wen-lu/msa_1b_mouse/20250710_lift_out/Position_10_2/'
P_BasePath = os.path.join(PasePath,'ts-aligned/imod_2D/imod_subtomo_extract/particles_rec_rename/')
os.makedirs(os.path.join(PasePath,'averaging/'), exist_ok=True)
output_list = os.path.join(PasePath,'averaging/tau_particles_list_bin0_720.txt')

zaehler = 0
particles_list = []
k = 1
files = [f for f in os.listdir(P_BasePath) if not f.startswith('.')]
files.sort()

for i in range(len(files)):
    particles_list.append(os.path.join(P_BasePath, files[i]))
    zaehler += 1

# Create combined list
particles_list_combined = []
for p in particles_list:
    particles_list_combined.append(p)
    particles_list_combined.append('singleaxiswedge 30 80')
    particles_list_combined.append('allpass')
    particles_list_combined.append(' ')

# Write particle list file

placeholder = '0'
with open(output_list, 'w') as fid:
    for line in particles_list_combined:
        fid.write(line + '\n')

In [ ]:
# Project with gpu
import os
import cupy as cp      
import numpy as np     
import mrcfile
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import mrcfile
import warnings
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

def process_particle_gpu(args):
    """
    Process an MRC particle file using the specified GPU.

    Args:
        args (tuple): Contains (particle_path, projectPath, gpu_id)

    Steps:
    1. Reads the MRC file (CPU)
    2. Transfers data to GPU, crops and averages slices (GPU)
    3. Normalizes the projection (GPU)
    4. Transfers result back to CPU and writes it as an MRC file.
    """
    particle_path, projectPath, gpu_id = args
    particle_name = os.path.splitext(os.path.basename(particle_path))[0]
    out_mrc = os.path.join(projectPath, f"{particle_name}.mrc")

    # Set the GPU device for this task
    with cp.cuda.Device(gpu_id):
        # Read data on CPU
        with mrcfile.open(particle_path, mode='r') as mrc:
            particle_cpu = mrc.data  # NumPy array on CPU

        # Transfer data to GPU as float32
        particle_gpu = cp.asarray(particle_cpu, dtype=cp.float32)

        # GPU slicing: crop along the first axis [188:313, :, :]
        trimmed_particle = particle_gpu[188:313, :, :]

        # Compute the mean projection along axis 0
        proj_gpu = cp.mean(trimmed_particle, axis=0)

        # Normalize the projection on GPU
        mean_val = cp.mean(proj_gpu)
        std_val = cp.std(proj_gpu)
        proj_gpu = (proj_gpu - mean_val) / std_val

        # Transfer the result back to CPU memory as float32
        proj_cpu = proj_gpu.get().astype(np.float32)

    # Write the normalized projection as a new MRC file (CPU)
    with mrcfile.new(out_mrc, overwrite=True) as mrc_out:
        mrc_out.set_data(proj_cpu)

    print(f"Saved: {out_mrc}")
    # The string returned is written to the STAR file; adjust if needed.
    return f"{out_mrc}   {out_mrc}"

def generate_projections_gpu(particles_list, projectPath, star_filename):
    """
    Process a list of particle files concurrently, assign each process a GPU,
    and write out a STAR file with the result filenames.
    """
    os.makedirs(projectPath, exist_ok=True)
    num_workers = 4

    # Create task list with round-robin GPU assignment.
    tasks = [
        (particle, projectPath, i % num_workers)
        for i, particle in enumerate(particles_list)
    ]

    # Use a process pool with num_workers; each job explicitly sets its GPU device.
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        results = list(executor.map(process_particle_gpu, tasks))

    # Write STAR file
    with open(star_filename, 'w') as fid:
        fid.write("data_\n")
        fid.write("loop_\n")
        fid.write("_rlnImageName #1\n")
        fid.write("_rlnMicrographName #2\n")
        for line in results:
            fid.write(line + '\n')

def generate_projections_cpu(particles_list, projectPath, star_filename):
    os.makedirs(projectPath, exist_ok=True)
    
    # --- PERFORMANCE TUNING FOR 64 CORES ---
    # We use 60 workers to maximize throughput while leaving 4 cores
    # to manage the heavy Disk I/O traffic.
    num_workers = 60
    
    tasks = [
        (particle, projectPath)
        for particle in particles_list
    ]

    print(f"Starting processing of {len(tasks)} particles on {num_workers} CPU cores...")

    # Using chunksize=1 ensures that if one file is slow (disk lag), 
    # other fast cores can keep picking up new work instantly.
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        results = list(tqdm(
            executor.map(process_particle_cpu, tasks, chunksize=1), 
            total=len(tasks),
            unit="img",
            ncols=100
        ))

    # Filter out any failed files (None)
    valid_results = [r for r in results if r is not None]

    with open(star_filename, 'w') as fid:
        fid.write("data_\n")
        fid.write("loop_\n")
        fid.write("_rlnImageName #1\n")
        fid.write("_rlnMicrographName #2\n")
        for line in valid_results:
            fid.write(line + '\n')
            
    print(f"Done. STAR file written to {star_filename}")

# Example usage:
# Note: Make sure that 'ParseParticleList' is defined elsewhere in your code.
# decide if you want gpu or cpu to do projection
BasePath = '/mnt/storage/data/users/wen-lu/msa_1b_mouse/20250710_lift_out/Position_10_2/'
output_list = os.path.join(BasePath,'averaging/tau_particles_list_bin0_720.txt')
particles_list, skip, skip2, skip3 = ParseParticleList(output_list)
projectPath = os.path.join(BasePath,'Particles_Proj_720_z124/')
star_file = os.path.join(BasePath, ('averaging/tau_particles_list_bin0_720_proj_z124.star'))

os.makedirs(projectPath, exist_ok=True)
generate_projections_gpu(particles_list, projectPath, star_file)


In [ ]:
# Normalize with Relion preprocess (assuming relion is accessible) relion version 3.0.8_cu10.1, openmpi 2.1.2
import subprocess
subprocess.run(["mpirun", "-np", "48", "relion_preprocess_mpi",
                "--operate_on", "/mnt/storage/data/users/wen-lu/msa_1b_mouse/20250710_lift_out/Position_10_2/averaging/tau_particles_list_bin0_720_proj_z124.star",
                "--norm", "--bg_radius", "250",
                "--operate_out", "/mnt/storage/data/users/wen-lu/msa_1b_mouse/20250710_lift_out/Position_10_2/averaging/tau_particles_list_bin0_720_proj_z124_norm_inv",
                "--set_angpix", "2.42", "--invert_contrast"])

In [ ]:
# Prepare for the prior angle in 2D

# add the optic header to convert relion3 to relion5 

# data_optics

# loop_ 
# _rlnOpticsGroupName #1 
# _rlnOpticsGroup #2 
# _rlnMicrographOriginalPixelSize #3
# _rlnVoltage #4
# _rlnSphericalAberration #5 
# _rlnAmplitudeContrast #6 
# _rlnImagePixelSize #7 
# _rlnImageSize #8 
# _rlnImageDimensionality #9 
# opticsGroup1            1    2.42   300.000000     2.700000     0.070000     2.42           500        2 

# data_particles

# loop_ 
# _rlnImageName #1 
# _rlnMicrographName #2 
# _rlnOpticsGroup #3


import pandas as pd
import re
import os

# Function to read STAR file and separate header and data
def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df
    
def create_rlnHelicalTrackLengthAngst(helicaltubeid: pd.DataFrame, pick_distance: float) -> pd.DataFrame:
    """
    This function takes:
      - `helicaltubeid`: A pandas DataFrame that has a column 'HelicalTubeID' identifying each helical segment.
      - `pick_distance`: The interbox distance (in Å).

    It returns a new DataFrame containing, for each unique HelicalTubeID:
      - HelicalTubeID
      - StartDistance (always 0)
      - TotalDistance (in Å, computed as number_of_rows * pick_distance)
      - PickDistance (in Å, the same for all rows)

    Example:
        If tube_id 1 has 10 particles, and pick_distance=4.7Å,
        TotalDistance for tube_id=1 will be 10 * 4.7 = 47 Å.
    """
    df = helicaltubeid.copy()
    df = helicaltubeid.to_frame(name="HelicalTubeID")
    df['ParticleIndexWithinTube'] = df.groupby('HelicalTubeID').cumcount()
    df['DistanceFromStart'] = df['ParticleIndexWithinTube'] * pick_distance
    return df
    

# Paths to your STAR files
basepath = '/mnt/storage/data/users/wen-lu/msa_1b_mouse/20250710_lift_out/Position_10_2/'
warp_file = os.path.join(basepath,'warp/Position_10_2/particles_warp.star')
relion_file = os.path.join(basepath,'averaging/tau_particles_list_bin0_720_proj_z124_norm_inv.star')
output_file = os.path.join(basepath, 'averaging/tau_particles_list_bin0_720_proj_z124_norm_inv_relion5_new.star')


# Read Warp STAR, assuming 15 header lines (adjust if necessary)
warp_header, warp_df = read_star(warp_file, header_lines=14)

# Convert relevant columns to numeric
# Assuming columns (index-based) for psi_angle, tilt_angle, helicaltubeid:
# In your Warp header example, psi_angle is column 8, tilt_angle column 7, helicaltubeid column 6 (0-indexed as 7,6,5).
warp_df = warp_df.apply(pd.to_numeric, errors='ignore')
psi_angle = warp_df.iloc[:, 8]
tilt_angle = warp_df.iloc[:, 7]
helicaltubeid = warp_df.iloc[:, 6]
rlnCoordinateX = warp_df.iloc[:,1]
rlnCoordinateY = warp_df.iloc[:,2]
rlnCoordinateZ = warp_df.iloc[:,3]

# Read Relion STAR
relion_header, relion_df = read_star(relion_file, header_lines=7)

adjust_number = relion_df.iloc[:,1].str.extract(r'p_(\d+)\.mrc$', expand=False).astype(int)
adjust_number = adjust_number-1

relion_df = relion_df.apply(pd.to_numeric, errors='ignore')



aligned_psi = psi_angle.iloc[adjust_number]
aligned_psi = aligned_psi.reset_index(drop=True)
aligned_tilt = tilt_angle.iloc[adjust_number]
aligned_tilt = aligned_tilt.reset_index(drop=True)
aligned_helical = helicaltubeid.iloc[adjust_number]
aligned_helical = aligned_helical.reset_index(drop=True)
aligned_rlnCoordinateX = rlnCoordinateX.iloc[adjust_number]
aligned_rlnCoordinateX = aligned_rlnCoordinateX.reset_index(drop=True)
aligned_rlnCoordinateY = rlnCoordinateY.iloc[adjust_number]
aligned_rlnCoordinateY = aligned_rlnCoordinateY.reset_index(drop=True)
aligned_rlnCoordinateZ = rlnCoordinateZ.iloc[adjust_number]
aligned_rlnCoordinateZ = aligned_rlnCoordinateZ.reset_index(drop=True)



relion_df['_rlnOpticsGroup'] = 1
relion_df['_rlnAnglePsiPrior'] = aligned_psi.values
relion_df['_rlnAngleTiltPrior'] = aligned_tilt
relion_df['_rlnHelicalTubeID'] = aligned_helical.values
relion_df['_rlnAnglePsiFlipRatio'] = 0.5
relion_df['_rlnCoordinateX'] = aligned_rlnCoordinateX.values
relion_df['_rlnCoordinateY'] = aligned_rlnCoordinateY.values
relion_df['_rlnCoordinateZ'] = aligned_rlnCoordinateZ.values

helicaltubeid = relion_df['_rlnHelicalTubeID']
inter_box_distance = 4
pixel_size = 2.42
bin_size = 4
HelicalTrackLengthAngst = create_rlnHelicalTrackLengthAngst(helicaltubeid,pixel_size*bin_size*inter_box_distance)
relion_df['_rlnHelicalTrackLengthAngst'] = HelicalTrackLengthAngst['DistanceFromStart'].values

tomo_sizeX = 4096
tomo_sizeY = 4096
tomo_sizeZ = 3000

relion_df['_rlnCenteredCoordinateXAngst'] = (relion_df['_rlnCoordinateX']-tomo_sizeX/2)*pixel_size
relion_df['_rlnCenteredCoordinateYAngst'] = (relion_df['_rlnCoordinateY']-tomo_sizeY/2)*pixel_size
relion_df['_rlnCenteredCoordinateZAngst'] = (relion_df['_rlnCoordinateZ']-tomo_sizeZ/2)*pixel_size


# Update header to include the new column definitions.
# You need to find correct placement in header for new loop items.
# For simplicity, appending new definitions at the end of header.

new_header_lines = ["data_optics", "","loop_", "","_rlnOpticsGroupName #1", "_rlnOpticsGroup #2",
                    "_rlnMicrographOriginalPixelSize #3","_rlnVoltage #4","_rlnSphericalAberration #5","_rlnAmplitudeContrast #6"
                    ,"_rlnImagePixelSize #7","_rlnImageSize #8","_rlnImageDimensionality #9",
                    "opticsGroup1            1    2.42   300.000000     2.700000     0.070000     2.42           500        2","",
                    "data_particles", "","loop_","_rlnImageName #1","_rlnMicrographName #2","_rlnOpticsGroup #3"]
# new_header_lines = list(relion_header)  # Copy existing header lines
new_header_lines.append("_rlnAnglePsiPrior #4")
new_header_lines.append("_rlnAngleTiltPrior #5")
new_header_lines.append("_rlnHelicalTubeID #6")
new_header_lines.append("_rlnAnglePsiFlipRatio #7")
new_header_lines.append("_rlnCoordinateX #8")
new_header_lines.append("_rlnCoordinateY #9")
new_header_lines.append("_rlnCoordinateZ #10")
new_header_lines.append("_rlnHelicalTrackLengthAngst #11")
new_header_lines.append("_rlnCenteredCoordinateXAngst #12")
new_header_lines.append("_rlnCenteredCoordinateYAngst #13")
new_header_lines.append("_rlnCenteredCoordinateZAngst #14")

# Write the new STAR file

with open(output_file, 'w') as f:
    # Write header lines
    for line in new_header_lines:
        f.write(f"{line}\n")
    # Write a blank line to separate header and data if required
    f.write("\n")
    # Write dataframe to file using space as delimiter, without index and header
    relion_df.to_csv(f, sep=' ', index=False, header=False)


In [ ]:
# Prepare for from 2D select to 3D refine with random subset

import pandas as pd
import re

# Function to read STAR file and separate header and data
def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df
    
 # Paths to your STAR files
relion_file = '/mnt/storage/data3/users/wen-lu/12082/data/EMPIAR_deposition/CS2/warp/Position_28/relion_2D/Select/job005/particles.star'
output_file = '/mnt/storage/data3/users/wen-lu/12082/data/EMPIAR_deposition/CS2/warp/Position_28/relion_2D/Select/job005/particles_trimmed_3D.star'

# Read Relion STAR
relion_header, relion_df = read_star(relion_file, header_lines=49)
relion_df['_rlnRandomSubset'] = 0
# Assign random subset for each filament
filamentIDs = relion_df.iloc[:, 5].unique()
for fid in filamentIDs:
    mask = relion_df.iloc[:, 5] == fid  
    # Put the same filament coordinarte into the same group
    relion_df.loc[mask, relion_df.columns[26]] = int(fid)%2+1



# new_header_lines = list(relion_header)  # Copy existing header lines
# new_header_lines.append("_rlnRandomSubset #27")

# Write the new STAR file

with open(output_file, 'w') as f:
    # Write header lines
    for line in new_header_lines:
        f.write(f"{line}\n")
    # Write a blank line to separate header and data if required
    f.write("\n")
    # Write dataframe to file using space as delimiter, without index and header
    relion_df.to_csv(f, sep=' ', index=False, header=False)





In [ ]:
# pharsing data for dynamo after 2D clean-up


import pandas as pd
import numpy as np
import os

# Function to read STAR file and separate header and data
def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

def create_rlnHelicalTrackLengthAngst(helicaltubeid: pd.DataFrame, pick_distance: float) -> pd.DataFrame:
    """
    This function takes:
      - `helicaltubeid`: A pandas DataFrame that has a column 'HelicalTubeID' identifying each helical segment.
      - `pick_distance`: The interbox distance (in Å).

    It returns a new DataFrame containing, for each unique HelicalTubeID:
      - HelicalTubeID
      - StartDistance (always 0)
      - TotalDistance (in Å, computed as number_of_rows * pick_distance)
      - PickDistance (in Å, the same for all rows)

    Example:
        If tube_id 1 has 10 particles, and pick_distance=4.7Å,
        TotalDistance for tube_id=1 will be 10 * 4.7 = 47 Å.
    """
    df = helicaltubeid.copy()
    df = helicaltubeid.to_frame(name="HelicalTubeID")
    df['ParticleIndexWithinTube'] = df.groupby('HelicalTubeID').cumcount()
    df['DistanceFromStart'] = df['ParticleIndexWithinTube'] * pick_distance
    return df


# Paths to your STAR files
root_folder = '/mnt/storage/data3/users/wen-lu/tau/warp/position_8_5/'
twoD_file = os.path.join(root_folder,'relion_2D/Select/job018/particles.star')
warp_file = os.path.join(root_folder,'particles_warp.star')
threeD_file = os.path.join(root_folder, 'relion3_b8/particles.star')
output_file = os.path.join(root_folder, 'relion3_b8/particles_new_select.star')

pixel_size = 2.42
binning = 4
inter_box_distance = 1


# --- 1) Read 2D STAR and extract extra columns ---
twoD_header, twoD_df = read_star(twoD_file, header_lines=49)

# Pull out the numeric particleID and the rotation/translation columns
extrat = twoD_df.iloc[:, [1]].copy()
extrat.columns = ['orig_filename']



# from “_p_1234.mrc” → 1234
extrat['particleID'] = (
    extrat['orig_filename']
      .astype(str)
      .str.split('_p_').str[1]
      .str.split('.').str[0]
      .astype(int)
)


# read the original warp file

warp_header, warp_df = read_star(warp_file, header_lines=14)




warp_df['rot'] = 0
warp_df['tilt'] = warp_df.iloc[:,7]
warp_df['psi'] = warp_df.iloc[:,8]
warp_df['_rlnHelicalTubeID'] = warp_df.iloc[:,6]
helicaltubeid = warp_df['_rlnHelicalTubeID']
idx = extrat['particleID'].astype(int) - 1
HelicalTrack = create_rlnHelicalTrackLengthAngst(helicaltubeid,pixel_size*binning*inter_box_distance)
warp_df['HelicalTrackLengthAngst'] = HelicalTrack['DistanceFromStart']

# extrat['rot'] = 0
# extrat['tilt'] = 0
# extrat['psi']  = 0
extrat['rot'] = 0
extrat['tilt'] = warp_df.iloc[idx, 7].values
extrat['psi']  = warp_df.iloc[idx, 8].values
extrat['_rlnHelicalTubeID']  = warp_df.iloc[idx, 6].values
extrat['HelicalTrackLengthAngst'] = warp_df['HelicalTrackLengthAngst']


# --- 2) Read 3D STAR and select matching rows ---
threeD_header, threeD_df = read_star(threeD_file, header_lines=16)

# assume column 1 is the “orig_filename” that embeds the particle number
threeD_df['relion_particlenumber'] = threeD_df.iloc[:, 8].astype(str).str.split('_ali_').str[1].str.split('_ctf_').str[0].astype(int)
threeD_df['relion_particlenumber'] = threeD_df['relion_particlenumber'] + 1


# keep only those in the 2D list
mask = threeD_df['relion_particlenumber'].isin(extrat['particleID'])
new_3D_df = threeD_df[mask].copy()

# --- 3) Merge in the 2D rotations/translations by particleID ---
new_3D_df = new_3D_df.merge(
    extrat[['particleID', 'rot', 'tilt', 'psi', '_rlnHelicalTubeID', 'HelicalTrackLengthAngst']],
    left_on='relion_particlenumber',
    right_on='particleID',
    how='left',
    sort=False
)

# drop helpers before writing
new_3D_df = new_3D_df.drop(
    columns=['relion_particlenumber', 'particleID'])


# Update header to include the new column definitions.
# You need to find correct placement in header for new loop items.
# For simplicity, appending new definitions at the end of header.
new_header_lines = list(threeD_header)  # Copy existing header lines
new_header_lines.append("_rlnAngleRot #13")
new_header_lines.append("_rlnAngleTilt #14")
new_header_lines.append("_rlnAnglePsi #15")
new_header_lines.append("_rlnHelicalTubeID #16")
new_header_lines.append("_rlnHelicalTrackLengthAngst #17")

# Adjust the numbering (#16, #17, #18) if needed based on your file structure.

# Write the new STAR file
with open(output_file, 'w') as f:
    # Write header lines
    for line in new_header_lines:
        f.write(line.rstrip("\n") + "\n")
    # Write a blank line to separate header and data if required

    # Write dataframe to file using space as delimiter, without index and header
    new_3D_df.to_csv(f, sep=' ', index=False, header=False)

In [ ]:
# prepare star file without 2D

import pandas as pd
import numpy as np
import os

# Function to read STAR file and separate header and data
def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

def create_rlnHelicalTrackLengthAngst(helicaltubeid: pd.DataFrame, pick_distance: float) -> pd.DataFrame:
    """
    This function takes:
      - `helicaltubeid`: A pandas DataFrame that has a column 'HelicalTubeID' identifying each helical segment.
      - `pick_distance`: The interbox distance (in Å).

    It returns a new DataFrame containing, for each unique HelicalTubeID:
      - HelicalTubeID
      - StartDistance (always 0)
      - TotalDistance (in Å, computed as number_of_rows * pick_distance)
      - PickDistance (in Å, the same for all rows)

    Example:
        If tube_id 1 has 10 particles, and pick_distance=4.7Å,
        TotalDistance for tube_id=1 will be 10 * 4.7 = 47 Å.
    """
    df = helicaltubeid.copy()
    df = helicaltubeid.to_frame(name="HelicalTubeID")
    df['ParticleIndexWithinTube'] = df.groupby('HelicalTubeID').cumcount()
    df['DistanceFromStart'] = df['ParticleIndexWithinTube'] * pick_distance
    return df


# Paths to your STAR files
root_folder = '/mnt/storage/data3/users/wen-lu/tau/warp/position_8_5/'
twoD_file = os.path.join(root_folder,'relion_2D/Select/job018/particles.star')
warp_file = os.path.join(root_folder,'particles_warp.star')
threeD_file = os.path.join(root_folder, 'relion3_b8/particles.star')
output_file = os.path.join(root_folder, 'relion3_b8/particles_new_select.star')

pixel_size = 2.42
binning = 4
inter_box_distance = 1


# --- 1) Read 2D STAR and extract extra columns ---
twoD_header, twoD_df = read_star(twoD_file, header_lines=49)

# Pull out the numeric particleID and the rotation/translation columns
extrat = twoD_df.iloc[:, [1]].copy()
extrat.columns = ['orig_filename']



# from “_p_1234.mrc” → 1234
extrat['particleID'] = (
    extrat['orig_filename']
      .astype(str)
      .str.split('_p_').str[1]
      .str.split('.').str[0]
      .astype(int)
)


# read the original warp file

warp_header, warp_df = read_star(warp_file, header_lines=14)




warp_df['rot'] = 0
warp_df['tilt'] = warp_df.iloc[:,7]
warp_df['psi'] = warp_df.iloc[:,8]
warp_df['_rlnHelicalTubeID'] = warp_df.iloc[:,6]
helicaltubeid = warp_df['_rlnHelicalTubeID']
idx = extrat['particleID'].astype(int) - 1
HelicalTrack = create_rlnHelicalTrackLengthAngst(helicaltubeid,pixel_size*binning*inter_box_distance)
warp_df['HelicalTrackLengthAngst'] = HelicalTrack['DistanceFromStart']

# extrat['rot'] = 0
# extrat['tilt'] = 0
# extrat['psi']  = 0
extrat['rot'] = 0
extrat['tilt'] = warp_df.iloc[idx, 7].values
extrat['psi']  = warp_df.iloc[idx, 8].values
extrat['_rlnHelicalTubeID']  = warp_df.iloc[idx, 6].values
extrat['HelicalTrackLengthAngst'] = warp_df['HelicalTrackLengthAngst']


# --- 2) Read 3D STAR and select matching rows ---
threeD_header, threeD_df = read_star(threeD_file, header_lines=16)

# assume column 1 is the “orig_filename” that embeds the particle number
threeD_df['relion_particlenumber'] = threeD_df.iloc[:, 8].astype(str).str.split('_ali_').str[1].str.split('_ctf_').str[0].astype(int)
threeD_df['relion_particlenumber'] = threeD_df['relion_particlenumber'] + 1


# keep only those in the 2D list
mask = threeD_df['relion_particlenumber'].isin(extrat['particleID'])
new_3D_df = threeD_df[mask].copy()

# --- 3) Merge in the 2D rotations/translations by particleID ---
new_3D_df = new_3D_df.merge(
    extrat[['particleID', 'rot', 'tilt', 'psi', '_rlnHelicalTubeID', 'HelicalTrackLengthAngst']],
    left_on='relion_particlenumber',
    right_on='particleID',
    how='left',
    sort=False
)

# drop helpers before writing
new_3D_df = new_3D_df.drop(
    columns=['relion_particlenumber', 'particleID'])


# Update header to include the new column definitions.
# You need to find correct placement in header for new loop items.
# For simplicity, appending new definitions at the end of header.
new_header_lines = list(threeD_header)  # Copy existing header lines
new_header_lines.append("_rlnAngleRot #13")
new_header_lines.append("_rlnAngleTilt #14")
new_header_lines.append("_rlnAnglePsi #15")
new_header_lines.append("_rlnHelicalTubeID #16")
new_header_lines.append("_rlnHelicalTrackLengthAngst #17")

# Adjust the numbering (#16, #17, #18) if needed based on your file structure.

# Write the new STAR file
with open(output_file, 'w') as f:
    # Write header lines
    for line in new_header_lines:
        f.write(line.rstrip("\n") + "\n")
    # Write a blank line to separate header and data if required

    # Write dataframe to file using space as delimiter, without index and header
    new_3D_df.to_csv(f, sep=' ', index=False, header=False)

In [ ]:
# prepare .tbl file

import os
import glob
import subprocess
import numpy as np
import pandas as pd


def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

basepath = '/mnt/storage/data/users/lukas/20251127_MSA_liftout/Session1/msa_human/RelionProcessing/Position_47_2/warp/'
starfile_h = os.path.join(basepath, 'relion3_b8/particles_new_select.star')
# 1) Run warp2dynamo
subprocess.run(
    [
        'warp2dynamo',
        '-i', os.path.join(basepath, 'relion3_b8/particles_new_select.star'),
        '-o', os.path.join(basepath, 'dynamo/particles'),
        '-bs', '64'
    ],
    check=True
)
tomostar_glob = os.path.join(basepath, 'processing', 'tomostar', '*.tomostar')
tomostar_paths = sorted(glob.glob(tomostar_glob))
if not tomostar_paths:
    raise FileNotFoundError(f'No .tomostar files found matching: {tomostar_glob}')
tomostar_path = tomostar_paths[0]

# 3) Load particle table (as strings)
# edit the .tbl file for tomogram number (#13), group number (#20), and missing wedge angle (# 14 #15) at the coreect position
particle_table = np.loadtxt(os.path.join(basepath,'dynamo/particles.tbl'), comments='#', dtype=str)


_, tilt_star_df = read_star(tomostar_path, header_lines=10)
_, h_df = read_star(starfile_h, header_lines=21)


tilt = -(tilt_star_df.iloc[:, 1].astype(float).to_numpy())
tilt_min = float(np.min(tilt))
tilt_max = float(np.max(tilt))
edit_particle_table = particle_table.copy()
edit_particle_table[:, [12, 13, 14, 19]] = np.array(
    [str(1), f'{tilt_min:.6f}', f'{tilt_max:.6f}', str(1)],
    dtype=object
)

# 6 add filament number in the column 23 and reorder for better grouping
helical_id = h_df.iloc[:,15].astype(float).to_numpy()
edit_particle_table[:, 22] = helical_id


# 7) Save edited table
output_table = os.path.join(basepath, 'dynamo', 'particles_edit.tbl')
np.savetxt(output_table, edit_particle_table, delimiter=' ', fmt='%s')
print(f'Edited table written to: {output_table}')






Usage: warp2dynamo [OPTIONS]
Try 'warp2dynamo --help' for help.

Error: Invalid value for '--input_star_file' / '-i': Path '/mnt/storage/data/users/lukas/20251127_MSA_liftout/Session1/msa_human/RelionProcessing/Position_47_2/warp/relion3_b8/particles_new_select.star' does not exist.


CalledProcessError: Command '['warp2dynamo', '-i', '/mnt/storage/data/users/lukas/20251127_MSA_liftout/Session1/msa_human/RelionProcessing/Position_47_2/warp/relion3_b8/particles_new_select.star', '-o', '/mnt/storage/data/users/lukas/20251127_MSA_liftout/Session1/msa_human/RelionProcessing/Position_47_2/warp/dynamo/particles', '-bs', '64']' returned non-zero exit status 2.

In [ ]:
## !/usr/bin/env python3
# make particles.tbl from warp star file and reconstructed .mrc file
# ------------------------------------------------------------------

import os, subprocess, numpy as np, pandas as pd, mrcfile
from multiprocessing import Pool, cpu_count

def bandpass_filter(vol, pixel_size, low_f, high_f):
    """
    Apply an ideal 3D band-pass filter to `vol`.
    - vol: 3D numpy array
    - pixel_size: in Å
    - low_f: minimum frequency (1/Å)
    - high_f: maximum frequency (1/Å)
    """
    dims = vol.shape
    # build frequency axes for each dimension
    freq_axes = [np.fft.fftfreq(n, d=ps) for n, ps in zip(dims, [pixel_size]*3)]
    fx, fy, fz = np.meshgrid(*freq_axes, indexing='ij')
    radius = np.sqrt(fx**2 + fy**2 + fz**2)

    # ideal band-pass mask
    mask = (radius >= low_f) & (radius <= high_f)

    # forward FFT, apply mask, inverse FFT
    vol_fft = np.fft.fftn(vol)
    vol_fft *= mask
    filtered = np.fft.ifftn(vol_fft)

    return np.real(filtered)

# ------------ helpers --------------------------------------------------------
def write_em_via_mrc(tmp_mrc: str, out_em: str):
    """Call EMAN2 to convert an MRC volume to EM."""
    subprocess.run(['e2proc3d.py', '--mult=-1', tmp_mrc, out_em], check=True, stdout=subprocess.DEVNULL)

def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

# ------------ configuration --------------------------------------------------

pixel_size = 19.36
low_f = 1/500
high_f = 1/2
do_ctf_correction = 1 

tomo_root_path = ['/mnt/storage/data3/users/wen-lu/tau/warp/position_8_5/']
for root_path in tomo_root_path:
    warp_file      = os.path.join(root_path,'relion3_b8/particles_new_select.star')
    tbl_file       = os.path.join(root_path, 'dynamo/particles_edit.tbl')
    warp_root_path = os.path.join(root_path, 'relion3_b8/')
    outputfolder   = os.path.join(root_path,'dynamo/filamentsData_ctf/')
    do_ctf_correction = 1
    ctf_correction_method = 'wiener' # Options: 'phase_flip' or 'wiener'
    wiener_epsilon = 0.1

    os.makedirs(outputfolder, exist_ok=True)
    
    # ------------ pre-processing (single-thread) --------------------------------- might change!!
    _, warp_df = read_star(warp_file, header_lines=21)
    
    # extract relion particle numbers
    warp_df['relion_particlenumber'] = (
        warp_df.iloc[:, 8]
            .astype(str)
            .str.split('_ali_').str[1]
            .str.split('_ctf_').str[0]
            .astype(int)
    )
    
    par_table = np.loadtxt(tbl_file, comments='#', dtype=float)
    par_table = par_table[np.argsort(par_table[:, 0])]
    particle_mask_idx = par_table[:, 0].astype(int) - 1          # zero-based
    
    mrc_particle_list = warp_df.iloc[particle_mask_idx, 7].values
    ctf_list          = warp_df.iloc[particle_mask_idx, 8].values
    
    tasks = [(idx, mrc, ctf, int(par_table[idx, 0])) for idx, (mrc, ctf) in enumerate(zip(mrc_particle_list, ctf_list))]
    
    # ------------ worker ---------------------------------------------------------
    def convert_particle(task):
        """Run on a single particle (execution happens in forked worker)."""
        i, mrc_name, ctf_name, particle_num = task
        in_mrc  = os.path.join(warp_root_path, mrc_name)
        in_ctf = os.path.join(warp_root_path, ctf_name)
        out_em  = os.path.join(outputfolder, f'particle_{particle_num:06d}.em')
        out_ctf = os.path.join(outputfolder, f'pfmask_{particle_num:06d}.em')
    
        if do_ctf_correction == 1:
            with mrcfile.open(in_mrc, permissive=True) as m:
                subtomo = m.data.astype(np.float32)
            dim = subtomo.shape
            with mrcfile.open(in_ctf, permissive=True) as m:
                ctf_vol = m.data.astype(np.float32)
            
            expected_dim = (dim[0], dim[1], dim[2]//2+1)
            
            if ctf_vol.shape == expected_dim:  # half-spectrum
                subtomo_fft = np.fft.rfftn(subtomo)
                
                if ctf_correction_method == 'phase_flip':
                    ctf_filter = np.sign(ctf_vol)
                elif ctf_correction_method == 'wiener':
                    ctf_filter = ctf_vol / (ctf_vol**2 + wiener_epsilon)
                else:
                    raise ValueError(f"Unknown method: {ctf_correction_method}")
                
                corrected = np.fft.irfftn(subtomo_fft * ctf_filter, s=dim)
                
            else:  # full-spectrum
                subtomo_fft = np.fft.fftn(subtomo)
                
                if ctf_correction_method == 'phase_flip':
                    ctf_filter = np.sign(ctf_vol)
                elif ctf_correction_method == 'wiener':
                    ctf_filter = ctf_vol / (ctf_vol**2 + wiener_epsilon)
                else:
                    raise ValueError(f"Unknown method: {ctf_correction_method}")
                
                corrected = np.real(np.fft.ifftn(subtomo_fft * ctf_filter))
        
                
            corrected = bandpass_filter(corrected, pixel_size, low_f, high_f)         
            tmp_mrc = os.path.join(outputfolder, f'_tmp_{particle_num:06d}.mrc')
            with mrcfile.new(tmp_mrc, overwrite=True) as m:
                m.set_data(corrected.astype(np.float32))
        
            write_em_via_mrc(tmp_mrc, out_em)
            os.remove(tmp_mrc)
               
        elif do_ctf_correction == 0:
            write_em_via_mrc(in_mrc, out_em)
        
        return particle_num                              # for progress reporting
    
    # ------------ parallel driver -----------------------------------------------
    if __name__ == '__main__':
        nproc = min(cpu_count(), 64)          # cap if you want; else cpu_count()
        print(f'Launching {nproc} workers for {len(tasks):,} particles…')
    
        with Pool(processes=nproc) as pool:
            for done_idx in pool.imap_unordered(convert_particle, tasks, chunksize=64):
                if done_idx % 100 == 0:       # lightweight progress ping
                    print(f'  finished particle {done_idx}')
    
        print('ALL DONE.')

In [ ]:
# edit the tble file for better grouping and S/N of abp job

import os
import glob
import subprocess
import numpy as np
import pandas as pd


basepath = '/mnt/storage/data3/users/wen-lu/tau/warp/position_8_5/'


# 3) Load particle table (as strings)
# edit the .tbl file for tomogram number (#13), group number (#20), and missing wedge angle (# 14 #15) at the coreect position
particle_table = np.loadtxt(os.path.join(basepath,'dynamo/dynamo_project/refined_table_ref_001_ite_0006.tbl'), comments='#', dtype=str)

edit_particle_table = particle_table

# 6 add filament number in the column 23 and reorder for better grouping
helical_id = particle_table[:, 22]

# Count particles per filament
unique_ids, counts = np.unique(helical_id, return_counts=True)
filament_counts = sorted(zip(unique_ids, counts), key=lambda x: x[1], reverse=True)

# Assign filaments to groups to balance sizes
group1_ids = []
group2_ids = []
group1_count = 0
group2_count = 0

for filament_id, count in filament_counts:
    if group1_count <= group2_count:
        group1_ids.append(filament_id)
        group1_count += count
    else:
        group2_ids.append(filament_id)
        group2_count += count

# Create masks for each group
group1_mask = np.isin(helical_id, group1_ids)
group2_mask = np.isin(helical_id, group2_ids)

# Get particles for each group
group1_particles = edit_particle_table[group1_mask]
group2_particles = edit_particle_table[group2_mask]

# Interleave: 1/2/1/2/1/2...
max_len = max(len(group1_particles), len(group2_particles))
interleaved_list = []

for i in range(max_len):
    if i < len(group1_particles):
        interleaved_list.append(group1_particles[i])
    if i < len(group2_particles):
        interleaved_list.append(group2_particles[i])

edit_particle_table = np.array(interleaved_list)



# 7) Save edited table
output_table = os.path.join(basepath, 'dynamo', 'dynamo_project', 'refined_table_ref_001_ite_0006_mod.tbl')
np.savetxt(output_table, edit_particle_table, delimiter=' ', fmt='%s')
print(f'Edited table written to: {output_table}')

In [ ]:
# Generating star files for warp in bin4

import numpy as np
import subprocess
import os
import pandas as pd
import matplotlib.pyplot as plt

def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

def create_rlnHelicalTrackLengthAngst(helicaltubeid: pd.DataFrame, pick_distance: float) -> pd.DataFrame:
    """
    This function takes:
      - `helicaltubeid`: A pandas DataFrame that has a column 'HelicalTubeID' identifying each helical segment.
      - `pick_distance`: The interbox distance (in Å).

    It returns a new DataFrame containing, for each unique HelicalTubeID:
      - HelicalTubeID
      - StartDistance (always 0)
      - TotalDistance (in Å, computed as number_of_rows * pick_distance)
      - PickDistance (in Å, the same for all rows)

    Example:
        If tube_id 1 has 10 particles, and pick_distance=4.7Å,
        TotalDistance for tube_id=1 will be 10 * 4.7 = 47 Å.
    """
    df = helicaltubeid.copy()
    df = helicaltubeid.to_frame(name="HelicalTubeID")
    df['ParticleIndexWithinTube'] = df.groupby('HelicalTubeID').cumcount()
    df['DistanceFromStart'] = df['ParticleIndexWithinTube'] * pick_distance
    return df

tomo_root_path = ['/mnt/storage/data3/users/wen-lu/tau/warp/position_8_5/']
dynamo_root_path = tomo_root_path[0] + 'dynamo/'
particle_table1 = os.path.join(dynamo_root_path,'dynamo_project/abp_align_eo/results/ite_0004/averages/refined_table_ref_001_ite_0004.tbl')
particle_table2 = os.path.join(dynamo_root_path,'dynamo_project/abp_align_eo/results/ite_0004/averages/refined_table_ref_002_ite_0004.tbl')
warp_ref = [os.path.join(root, 'particles_warp.star') for root in tomo_root_path]

bin_factor = 8
binning = 4 # this is the bin of picking
inter_box_distance = 1
pixel_size = 2.42

# get particle index, corrected xyz coordinates, and euler angles for each tomogram

par_table = np.concatenate((np.loadtxt(particle_table1, comments='#', dtype=str),np.loadtxt(particle_table2, comments='#', dtype=str)),axis=0)
par_table = par_table[np.argsort(par_table[:, 0].astype(int))]
tomo_indx = np.unique(par_table[:,19])

begin_number = 0
# save temp splited table, convert to star for warp, and delet the temp table
for indx in tomo_indx:
    current_root_path = tomo_root_path[indx.astype(int)-1]

    tomo_table = par_table[par_table[:, 19] == indx, :]
    table_temp_path = os.path.join(dynamo_root_path,'dynamo_project/abp_align_eo/results/ite_0004/averages/temp.tbl')
    np.savetxt(table_temp_path,tomo_table,delimiter=' ',fmt='%s')
    subprocess.run(['dynamo2warp','-i',table_temp_path,'-tm',os.path.join(current_root_path,'dynamo/particles.reextract.doc'),
                    '-o', os.path.join(current_root_path,'temp.star')])
    
    warp_file = os.path.join(current_root_path,'temp.star')
    warp_helical = os.path.join(current_root_path,'relion3_b8/particles_new_select.star')
    warp_rest_naming = os.path.join(current_root_path,'relion3_b8/particles_new_select.star')
    output_star = os.path.join(current_root_path,'particles_dynamo_b4.star')

    warp_header, warp_df = read_star(warp_file, header_lines=12)
    warp_3d_header, warp_3d_df = read_star(warp_rest_naming, header_lines=21)
    warp_helical_header, warp_helical_df = read_star(warp_helical, header_lines=21)
    particle_mask = (tomo_table[:, 0].astype(int)-begin_number) - 1
    warp_df.iloc[:,0:3] = warp_df.iloc[:,0:3].astype(float)*bin_factor
    warp_df.iloc[:,6] = warp_helical_df.iloc[0,3] # name
    warp_df['mag'] = warp_helical_df.iloc[0,4]
    warp_df['pixel'] = warp_helical_df.iloc[0,5]
    warp_df['helicalID'] = warp_helical_df.iloc[particle_mask,15].values
    warp_df['psi_flip'] = 0.5 # psi flip
    warp_df['HelicalTrackLengthAngst'] = warp_helical_df.iloc[particle_mask,16].values
    
    new_header_lines = list(warp_header)  # Copy existing header lines
    new_header_lines.append("_rlnMagnification #8")
    new_header_lines.append("_rlnDetectorPixelSize #9")
    new_header_lines.append("_rlnHelicalTubeID #10")
    new_header_lines.append("_rlnAnglePsiFlipRatio #11")
    new_header_lines.append("_rlnHelicalTrackLengthAngst #12")

    
    # Write the new STAR file
    with open(output_star, 'w') as f:
        # Write header lines
        for line in new_header_lines:
            f.write(line.rstrip("\n") + "\n")
        # Write a blank line to separate header and data if required
    
        # Write dataframe to file using space as delimiter, without index and header
        warp_df.to_csv(f, sep=' ', index=False, header=False)
    
    begin_number = begin_number + np.max(tomo_table[:,0].astype(int))
    os.remove(warp_file)
    os.remove(table_temp_path)



In [ ]:
# prepare .tbl file at bin4

import os
import glob
import subprocess
import numpy as np
import pandas as pd


def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

basepath = '/mnt/storage/data3/users/wen-lu/tau/warp/position_31/'
starfile_h = os.path.join(basepath, 'particles_dynamo_b4.star')
# 1) Run warp2dynamo
subprocess.run(
    [
        'warp2dynamo',
        '-i', os.path.join(basepath, 'relion3_b4/particles.star'),
        '-o', os.path.join(basepath, 'dynamo/particles_b4'),
        '-bs', '64'
    ],
    check=True
)
tomostar_glob = os.path.join(basepath, 'processing', 'tomostar', '*.tomostar')
tomostar_paths = sorted(glob.glob(tomostar_glob))
if not tomostar_paths:
    raise FileNotFoundError(f'No .tomostar files found matching: {tomostar_glob}')
tomostar_path = tomostar_paths[0]

# 3) Load particle table (as strings)
# edit the .tbl file for tomogram number (#13), group number (#20), and missing wedge angle (# 14 #15) at the coreect position
particle_table = np.loadtxt(os.path.join(basepath,'dynamo/particles_b4.tbl'), comments='#', dtype=str)


_, tilt_star_df = read_star(tomostar_path, header_lines=10)
_, h_df = read_star(starfile_h, header_lines=17)


tilt = -(tilt_star_df.iloc[:, 1].astype(float).to_numpy())
tilt_min = float(np.min(tilt))
tilt_max = float(np.max(tilt))
edit_particle_table = particle_table.copy()
edit_particle_table[:, [12, 13, 14, 19]] = np.array(
    [str(1), f'{tilt_min:.6f}', f'{tilt_max:.6f}', str(1)],
    dtype=object
)

# 6 add filament number in the column 23 and reorder for better grouping
helical_id = h_df.iloc[:,9].astype(float).to_numpy()
edit_particle_table[:, 22] = helical_id

# Count particles per filament
unique_ids, counts = np.unique(helical_id, return_counts=True)
filament_counts = sorted(zip(unique_ids, counts), key=lambda x: x[1], reverse=True)

# Assign filaments to groups to balance sizes
group1_ids = []
group2_ids = []
group1_count = 0
group2_count = 0

for filament_id, count in filament_counts:
    if group1_count <= group2_count:
        group1_ids.append(filament_id)
        group1_count += count
    else:
        group2_ids.append(filament_id)
        group2_count += count

# Create masks for each group
group1_mask = np.isin(helical_id, group1_ids)
group2_mask = np.isin(helical_id, group2_ids)

# Get particles for each group
group1_particles = edit_particle_table[group1_mask]
group2_particles = edit_particle_table[group2_mask]

# Interleave: 1/2/1/2/1/2...
max_len = max(len(group1_particles), len(group2_particles))
interleaved_list = []

for i in range(max_len):
    if i < len(group1_particles):
        interleaved_list.append(group1_particles[i])
    if i < len(group2_particles):
        interleaved_list.append(group2_particles[i])

edit_particle_table = np.array(interleaved_list)




# 7) Save edited table
output_table = os.path.join(basepath, 'dynamo', 'particles_b4_edit.tbl')
np.savetxt(output_table, edit_particle_table, delimiter=' ', fmt='%s')
print(f'Edited table written to: {output_table}')






In [ ]:
#!/usr/bin/env python3
# make particles.tbl from warp star file and reconstructed .mrc file
# ------------------------------------------------------------------

import os, subprocess, numpy as np, pandas as pd, mrcfile
from multiprocessing import Pool, cpu_count

def bandpass_filter(vol, pixel_size, low_f, high_f):
    """
    Apply an ideal 3D band-pass filter to `vol`.
    - vol: 3D numpy array
    - pixel_size: in Å
    - low_f: minimum frequency (1/Å)
    - high_f: maximum frequency (1/Å)
    """
    dims = vol.shape
    # build frequency axes for each dimension
    freq_axes = [np.fft.fftfreq(n, d=ps) for n, ps in zip(dims, [pixel_size]*3)]
    fx, fy, fz = np.meshgrid(*freq_axes, indexing='ij')
    radius = np.sqrt(fx**2 + fy**2 + fz**2)

    # ideal band-pass mask
    mask = (radius >= low_f) & (radius <= high_f)

    # forward FFT, apply mask, inverse FFT
    vol_fft = np.fft.fftn(vol)
    vol_fft *= mask
    filtered = np.fft.ifftn(vol_fft)

    return np.real(filtered)

# ------------ helpers --------------------------------------------------------
def write_em_via_mrc(tmp_mrc: str, out_em: str):
    """Call EMAN2 to convert an MRC volume to EM."""
    subprocess.run(['e2proc3d.py', '--mult=-1', tmp_mrc, out_em], check=True, stdout=subprocess.DEVNULL)

def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

# ------------ configuration --------------------------------------------------

pixel_size = 9.68
low_f = 1/500
high_f = 1/2
do_ctf_correction = 1 

tomo_root_path = ['/mnt/storage/data3/users/wen-lu/tau/warp/position_31/']
for root_path in tomo_root_path:
    warp_file      = os.path.join(root_path,'relion3_b4/particles.star')
    tbl_file       = os.path.join(root_path, 'dynamo/particles_b4_edit.tbl')
    warp_root_path = os.path.join(root_path, 'relion3_b4/')
    outputfolder   = os.path.join(root_path,'dynamo/filamentsData_b4_ctf/')
    do_ctf_correction = 1
    ctf_correction_method = 'wiener' # Options: 'phase_flip' or 'wiener'
    wiener_epsilon = 0.1

    os.makedirs(outputfolder, exist_ok=True)
    
    # ------------ pre-processing (single-thread) --------------------------------- might change!!
    _, warp_df = read_star(warp_file, header_lines=19)
    
    # extract relion particle numbers
    warp_df['relion_particlenumber'] = (
        warp_df.iloc[:, 11]
            .astype(str)
            .str.split('_ali_').str[1]
            .str.split('_ctf_').str[0]
            .astype(int)
    )
    
    par_table = np.loadtxt(tbl_file, comments='#', dtype=float)
    par_table = par_table[np.argsort(par_table[:, 0])]      
    particle_mask_idx = par_table[:, 0].astype(int) - 1          # zero-based
    mrc_particle_list = warp_df.iloc[particle_mask_idx, 10].values
    ctf_list          = warp_df.iloc[particle_mask_idx, 11].values
    
    tasks = [(idx, mrc, ctf) for idx, (mrc, ctf) in enumerate(zip(mrc_particle_list, ctf_list))]
    
    # ------------ worker ---------------------------------------------------------
    def convert_particle(task):
        """Run on a single particle (execution happens in forked worker)."""
        i, mrc_name, ctf_name = task
        in_mrc  = os.path.join(warp_root_path, mrc_name)
        in_ctf = os.path.join(warp_root_path, ctf_name)
        out_em  = os.path.join(outputfolder, f'particle_{i+1:06d}.em')
        out_ctf = os.path.join(outputfolder, f'pfmask_{i+1:06d}.em')
    
        if do_ctf_correction == 1:
            with mrcfile.open(in_mrc, permissive=True) as m:
                subtomo = m.data.astype(np.float32)
            dim = subtomo.shape
            with mrcfile.open(in_ctf, permissive=True) as m:
                ctf_vol = m.data.astype(np.float32)
            
            expected_dim = (dim[0], dim[1], dim[2]//2+1)
            
            if ctf_vol.shape == expected_dim:  # half-spectrum
                subtomo_fft = np.fft.rfftn(subtomo)
                
                if ctf_correction_method == 'phase_flip':
                    ctf_filter = np.sign(ctf_vol)
                elif ctf_correction_method == 'wiener':
                    ctf_filter = ctf_vol / (ctf_vol**2 + wiener_epsilon)
                else:
                    raise ValueError(f"Unknown method: {ctf_correction_method}")
                
                corrected = np.fft.irfftn(subtomo_fft * ctf_filter, s=dim)
                
            else:  # full-spectrum
                subtomo_fft = np.fft.fftn(subtomo)
                
                if ctf_correction_method == 'phase_flip':
                    ctf_filter = np.sign(ctf_vol)
                elif ctf_correction_method == 'wiener':
                    ctf_filter = ctf_vol / (ctf_vol**2 + wiener_epsilon)
                else:
                    raise ValueError(f"Unknown method: {ctf_correction_method}")
                
                corrected = np.real(np.fft.ifftn(subtomo_fft * ctf_filter))
        
                
            corrected = bandpass_filter(corrected, pixel_size, low_f, high_f)         
            tmp_mrc = os.path.join(outputfolder, f'_tmp_{i:06d}.mrc')
            with mrcfile.new(tmp_mrc, overwrite=True) as m:
                m.set_data(corrected.astype(np.float32))
        
            write_em_via_mrc(tmp_mrc, out_em)
            os.remove(tmp_mrc)
               
        elif do_ctf_correction == 0:
            write_em_via_mrc(in_mrc, out_em)
        
        return i                              # for progress reporting
    
    # ------------ parallel driver -----------------------------------------------
    if __name__ == '__main__':
        nproc = min(cpu_count(), 64)          # cap if you want; else cpu_count()
        print(f'Launching {nproc} workers for {len(tasks):,} particles…')
    
        with Pool(processes=nproc) as pool:
            for done_idx in pool.imap_unordered(convert_particle, tasks, chunksize=64):
                if done_idx % 100 == 0:       # lightweight progress ping
                    print(f'  finished particle {done_idx}')
    
        print('ALL DONE.')

In [ ]:
# Generating star files for warp in bin2

import numpy as np
import subprocess
import os
import pandas as pd
import matplotlib.pyplot as plt

def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

def create_rlnHelicalTrackLengthAngst(helicaltubeid: pd.DataFrame, pick_distance: float) -> pd.DataFrame:
    """
    This function takes:
      - `helicaltubeid`: A pandas DataFrame that has a column 'HelicalTubeID' identifying each helical segment.
      - `pick_distance`: The interbox distance (in Å).

    It returns a new DataFrame containing, for each unique HelicalTubeID:
      - HelicalTubeID
      - StartDistance (always 0)
      - TotalDistance (in Å, computed as number_of_rows * pick_distance)
      - PickDistance (in Å, the same for all rows)

    Example:
        If tube_id 1 has 10 particles, and pick_distance=4.7Å,
        TotalDistance for tube_id=1 will be 10 * 4.7 = 47 Å.
    """
    df = helicaltubeid.copy()
    df = helicaltubeid.to_frame(name="HelicalTubeID")
    df['ParticleIndexWithinTube'] = df.groupby('HelicalTubeID').cumcount()
    df['DistanceFromStart'] = df['ParticleIndexWithinTube'] * pick_distance
    return df

tomo_root_path = ['/mnt/storage/data3/users/wen-lu/tau/warp/position_9/']
dynamo_root_path = '/mnt/storage/data3/users/wen-lu/tau/warp/position_9/dynamo/'
particle_table1 = os.path.join(dynamo_root_path,'dynamo_project_b4/abp_align_eo/results/ite_0004/averages/refined_table_ref_001_ite_0004.tbl')
particle_table2 = os.path.join(dynamo_root_path,'dynamo_project_b4/abp_align_eo/results/ite_0004/averages/refined_table_ref_002_ite_0004.tbl')
warp_ref = [os.path.join(root, 'particles_warp.star') for root in tomo_root_path]

binning_factor = 4
binning = 4
inter_box_distance = 1

# get particle index, corrected xyz coordinates, and euler angles for each tomogram

par_table = np.concatenate((np.loadtxt(particle_table1, comments='#', dtype=str),np.loadtxt(particle_table2, comments='#', dtype=str)),axis=0)
par_table = par_table[np.argsort(par_table[:, 0].astype(int))]
tomo_indx = np.unique(par_table[:,19])

begin_number = 0
# save temp splited table, convert to star for warp, and delet the temp table
for indx in tomo_indx:
    current_root_path = tomo_root_path[indx.astype(int)-1]

    tomo_table = par_table[par_table[:, 19] == indx, :]
    table_temp_path = os.path.join(dynamo_root_path,'dynamo_project_b4/abp_align_eo/results/ite_0004/averages/temp.tbl')
    np.savetxt(table_temp_path,tomo_table,delimiter=' ',fmt='%s')
    subprocess.run(['dynamo2warp','-i',table_temp_path,'-tm',os.path.join(current_root_path,'dynamo/particles.reextract.doc'),
                    '-o', os.path.join(current_root_path,'temp.star')])
    
    warp_file = os.path.join(current_root_path,'temp.star')
    warp_helical = os.path.join(current_root_path,'particles_dynamo_b4.star')
    warp_rest_naming = os.path.join(current_root_path,'relion3_b4/particles.star')
    output_star = os.path.join(current_root_path,'particles_dynamo_b2.star')
    

    warp_header, warp_df = read_star(warp_file, header_lines=12)
    warp_3d_header, warp_3d_df = read_star(warp_rest_naming, header_lines=19)
    warp_helical_header, warp_helical_df = read_star(warp_helical, header_lines=17)

    warp_3d_df['particle_true_ID'] = warp_3d_df.iloc[:, 11].astype(str).str.split('_ali_').str[1].str.split('_ctf_').str[0].astype(int)
    particle_true_ids = warp_3d_df["particle_true_ID"].to_numpy()
    particle_mask = (tomo_table[:, 0].astype(int)-begin_number) - 1 


    warp_df.iloc[:,0:3] = warp_df.iloc[:,0:3].astype(float)*binning_factor
    warp_df.iloc[:,6] = warp_helical_df.iloc[0,6] # name
    warp_df['mag'] = warp_helical_df.iloc[0,7]
    warp_df['pixel'] = warp_helical_df.iloc[0,8]
    warp_df['helicalID'] = warp_helical_df.iloc[particle_mask,9].values
    warp_df['psi_flip'] = warp_helical_df.iloc[0,10] # psi flip
    warp_df['HelicalTrackLengthAngst'] = warp_helical_df.iloc[particle_mask,11].values
    
    new_header_lines = list(warp_header)  # Copy existing header lines
    new_header_lines.append("_rlnMagnification #8")
    new_header_lines.append("_rlnDetectorPixelSize #9")
    new_header_lines.append("_rlnHelicalTubeID #10")
    new_header_lines.append("_rlnAnglePsiFlipRatio #11")
    new_header_lines.append("_rlnHelicalTrackLengthAngst #12")
    
    # Write the new STAR file
    with open(output_star, 'w') as f:
        # Write header lines
        for line in new_header_lines:
            f.write(line.rstrip("\n") + "\n")
        # Write a blank line to separate header and data if required
    
        # Write dataframe to file using space as delimiter, without index and header
        warp_df.to_csv(f, sep=' ', index=False, header=False)
    
    begin_number = begin_number + np.max(tomo_table[:,0].astype(int))
    os.remove(warp_file)
    os.remove(table_temp_path)



In [ ]:
# prepare .tbl file at bin2

import os
import glob
import subprocess
import numpy as np
import pandas as pd


def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

basepath = '/mnt/storage/data3/users/wen-lu/tau/warp/position_9/'
starfile_h = os.path.join(basepath, 'particles_dynamo_b2.star')
# 1) Run warp2dynamo
subprocess.run(
    [
        'warp2dynamo',
        '-i', os.path.join(basepath, 'relion3_b2/particles.star'),
        '-o', os.path.join(basepath, 'dynamo/particles_b2'),
        '-bs', '64'
    ],
    check=True
)
tomostar_glob = os.path.join(basepath, 'processing', 'tomostar', '*.tomostar')
tomostar_paths = sorted(glob.glob(tomostar_glob))
if not tomostar_paths:
    raise FileNotFoundError(f'No .tomostar files found matching: {tomostar_glob}')
tomostar_path = tomostar_paths[0]

# 3) Load particle table (as strings)
# edit the .tbl file for tomogram number (#13), group number (#20), and missing wedge angle (# 14 #15) at the coreect position
particle_table = np.loadtxt(os.path.join(basepath,'dynamo/particles_b2.tbl'), comments='#', dtype=str)


_, tilt_star_df = read_star(tomostar_path, header_lines=10)
_, h_df = read_star(starfile_h, header_lines=17)


tilt = -(tilt_star_df.iloc[:, 1].astype(float).to_numpy())
tilt_min = float(np.min(tilt))
tilt_max = float(np.max(tilt))
edit_particle_table = particle_table.copy()
edit_particle_table[:, [12, 13, 14, 19]] = np.array(
    [str(1), f'{tilt_min:.6f}', f'{tilt_max:.6f}', str(1)],
    dtype=object
)

# 6 add filament number in the column 23 and reorder for better grouping
helical_id = h_df.iloc[:,9].astype(float).to_numpy()
edit_particle_table[:, 22] = helical_id

# Count particles per filament
unique_ids, counts = np.unique(helical_id, return_counts=True)
filament_counts = sorted(zip(unique_ids, counts), key=lambda x: x[1], reverse=True)

# Assign filaments to groups to balance sizes
group1_ids = []
group2_ids = []
group1_count = 0
group2_count = 0

for filament_id, count in filament_counts:
    if group1_count <= group2_count:
        group1_ids.append(filament_id)
        group1_count += count
    else:
        group2_ids.append(filament_id)
        group2_count += count

# Create masks for each group
group1_mask = np.isin(helical_id, group1_ids)
group2_mask = np.isin(helical_id, group2_ids)

# Get particles for each group
group1_particles = edit_particle_table[group1_mask]
group2_particles = edit_particle_table[group2_mask]

# Interleave: 1/2/1/2/1/2...
max_len = max(len(group1_particles), len(group2_particles))
interleaved_list = []

for i in range(max_len):
    if i < len(group1_particles):
        interleaved_list.append(group1_particles[i])
    if i < len(group2_particles):
        interleaved_list.append(group2_particles[i])

edit_particle_table = np.array(interleaved_list)




# 7) Save edited table
output_table = os.path.join(basepath, 'dynamo', 'particles_b2_edit.tbl')
np.savetxt(output_table, edit_particle_table, delimiter=' ', fmt='%s')
print(f'Edited table written to: {output_table}')






In [ ]:
#!/usr/bin/env python3
# make particles.tbl from warp star file and reconstructed .mrc file
# ------------------------------------------------------------------

import os, subprocess, numpy as np, pandas as pd, mrcfile
from multiprocessing import Pool, cpu_count

def bandpass_filter(vol, pixel_size, low_f, high_f):
    """
    Apply an ideal 3D band-pass filter to `vol`.
    - vol: 3D numpy array
    - pixel_size: in Å
    - low_f: minimum frequency (1/Å)
    - high_f: maximum frequency (1/Å)
    """
    dims = vol.shape
    # build frequency axes for each dimension
    freq_axes = [np.fft.fftfreq(n, d=pixel_size) for n, pixel_size in zip(dims, [pixel_size]*3)]
    fx, fy, fz = np.meshgrid(*freq_axes, indexing='ij')
    radius = np.sqrt(fx**2 + fy**2 + fz**2)

    # ideal band-pass mask
    mask = (radius >= low_f) & (radius <= high_f)

    # forward FFT, apply mask, inverse FFT
    vol_fft = np.fft.fftn(vol)
    vol_fft *= mask
    filtered = np.fft.ifftn(vol_fft)

    return np.real(filtered)

# ------------ helpers --------------------------------------------------------
def write_em_via_mrc(tmp_mrc: str, out_em: str):
    """Call EMAN2 to convert an MRC volume to EM."""
    subprocess.run(['e2proc3d.py', '--mult=-1', tmp_mrc, out_em], check=True, stdout=subprocess.DEVNULL)

def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

# ------------ configuration --------------------------------------------------

pixel_size = 4.84
low_f = 1/500
high_f = 1/2
do_ctf_correction = 1 

tomo_root_path = ['/mnt/storage/data3/users/wen-lu/tau/warp/position_9/']
for root_path in tomo_root_path:
    warp_file      = os.path.join(root_path,'relion3_b2/particles.star')
    tbl_file       = os.path.join(root_path, 'dynamo/particles_b2_edit.tbl')
    warp_root_path = os.path.join(root_path, 'relion3_b2/')
    outputfolder   = os.path.join(root_path,'dynamo/filamentsData_b2_ctf/')
    ctf_correction_method = 'wiener' # Options: 'phase_flip' or 'wiener'
    wiener_epsilon = 0.1

    os.makedirs(outputfolder, exist_ok=True)
    
    # ------------ pre-processing (single-thread) --------------------------------- might change!!
    _, warp_df = read_star(warp_file, header_lines=19)
    
    # extract relion particle numbers
    warp_df['relion_particlenumber'] = (
        warp_df.iloc[:, 11]
            .astype(str)
            .str.split('_ali_').str[1]
            .str.split('_ctf_').str[0]
            .astype(int)
    )
    
    par_table = np.loadtxt(tbl_file, comments='#', dtype=float)
    par_table = par_table[np.argsort(par_table[:, 0])] 
    particle_mask_idx = par_table[:, 0].astype(int) - 1          # zero-based
    
    mrc_particle_list = warp_df.iloc[particle_mask_idx, 10].values
    ctf_list          = warp_df.iloc[particle_mask_idx, 11].values
    
    tasks = [(idx, mrc, ctf) for idx, (mrc, ctf) in enumerate(zip(mrc_particle_list, ctf_list))]
    
    # ------------ worker ---------------------------------------------------------
    def convert_particle(task):
        """Run on a single particle (execution happens in forked worker)."""
        i, mrc_name, ctf_name = task
        in_mrc  = os.path.join(warp_root_path, mrc_name)
        in_ctf = os.path.join(warp_root_path, ctf_name)
        out_em  = os.path.join(outputfolder, f'particle_{i+1:06d}.em')
        out_ctf = os.path.join(outputfolder, f'pfmask_{i+1:06d}.em')
    
        if do_ctf_correction == 1:
            with mrcfile.open(in_mrc, permissive=True) as m:
                subtomo = m.data.astype(np.float32)
            dim = subtomo.shape
            with mrcfile.open(in_ctf, permissive=True) as m:
                ctf_vol = m.data.astype(np.float32)
            
            expected_dim = (dim[0], dim[1], dim[2]//2+1)
            
            if ctf_vol.shape == expected_dim:  # half-spectrum
                subtomo_fft = np.fft.rfftn(subtomo)
                
                if ctf_correction_method == 'phase_flip':
                    ctf_filter = np.sign(ctf_vol)
                elif ctf_correction_method == 'wiener':
                    ctf_filter = ctf_vol / (ctf_vol**2 + wiener_epsilon)
                else:
                    raise ValueError(f"Unknown method: {ctf_correction_method}")
                
                corrected = np.fft.irfftn(subtomo_fft * ctf_filter, s=dim)
                
            else:  # full-spectrum
                subtomo_fft = np.fft.fftn(subtomo)
                
                if ctf_correction_method == 'phase_flip':
                    ctf_filter = np.sign(ctf_vol)
                elif ctf_correction_method == 'wiener':
                    ctf_filter = ctf_vol / (ctf_vol**2 + wiener_epsilon)
                else:
                    raise ValueError(f"Unknown method: {ctf_correction_method}")
                
                corrected = np.real(np.fft.ifftn(subtomo_fft * ctf_filter))
        
                
            corrected = bandpass_filter(corrected, pixel_size, low_f, high_f)         
            tmp_mrc = os.path.join(outputfolder, f'_tmp_{i:06d}.mrc')
            with mrcfile.new(tmp_mrc, overwrite=True) as m:
                m.set_data(corrected.astype(np.float32))
        
            write_em_via_mrc(tmp_mrc, out_em)
            os.remove(tmp_mrc)
               
        elif do_ctf_correction == 0:
            write_em_via_mrc(in_mrc, out_em)
        
        return i                              # for progress reporting
    
    # ------------ parallel driver -----------------------------------------------
    if __name__ == '__main__':
        nproc = min(cpu_count(), 64)          # cap if you want; else cpu_count()
        print(f'Launching {nproc} workers for {len(tasks):,} particles…')
    
        with Pool(processes=nproc) as pool:
            for done_idx in pool.imap_unordered(convert_particle, tasks, chunksize=64):
                if done_idx % 100 == 0:       # lightweight progress ping
                    print(f'  finished particle {done_idx}')
    
        print('ALL DONE.')

In [ ]:
# Generating star files for warp in bin1

import numpy as np
import subprocess
import os
import pandas as pd
import matplotlib.pyplot as plt

def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df

def create_rlnHelicalTrackLengthAngst(helicaltubeid: pd.DataFrame, pick_distance: float) -> pd.DataFrame:
    """
    This function takes:
      - `helicaltubeid`: A pandas DataFrame that has a column 'HelicalTubeID' identifying each helical segment.
      - `pick_distance`: The interbox distance (in Å).

    It returns a new DataFrame containing, for each unique HelicalTubeID:
      - HelicalTubeID
      - StartDistance (always 0)
      - TotalDistance (in Å, computed as number_of_rows * pick_distance)
      - PickDistance (in Å, the same for all rows)

    Example:
        If tube_id 1 has 10 particles, and pick_distance=4.7Å,
        TotalDistance for tube_id=1 will be 10 * 4.7 = 47 Å.
    """
    df = helicaltubeid.copy()
    df = helicaltubeid.to_frame(name="HelicalTubeID")
    df['ParticleIndexWithinTube'] = df.groupby('HelicalTubeID').cumcount()
    df['DistanceFromStart'] = df['ParticleIndexWithinTube'] * pick_distance
    return df

tomo_root_path = ['/mnt/storage/data3/users/wen-lu/tau/warp/position_9/']
dynamo_root_path = '/mnt/storage/data3/users/wen-lu/tau/warp/position_9/dynamo/'
particle_table1 = os.path.join(dynamo_root_path,'dynamo_project_b2/abp_align_eo/results/ite_0004/averages/refined_table_ref_001_ite_0004.tbl')
particle_table2 = os.path.join(dynamo_root_path,'dynamo_project_b2/abp_align_eo/results/ite_0004/averages/refined_table_ref_002_ite_0004.tbl')
binning_factor = 2
binning = 4
inter_box_distance = 1

# get particle index, corrected xyz coordinates, and euler angles for each tomogram

par_table = np.concatenate((np.loadtxt(particle_table1, comments='#', dtype=str),np.loadtxt(particle_table2, comments='#', dtype=str)),axis=0)
par_table = par_table[np.argsort(par_table[:, 0].astype(int))]
tomo_indx = np.unique(par_table[:,19])

begin_number = 0
# save temp splited table, convert to star for warp, and delet the temp table
for indx in tomo_indx:
    current_root_path = tomo_root_path[indx.astype(int)-1]

    tomo_table = par_table[par_table[:, 19] == indx, :]
    table_temp_path = os.path.join(dynamo_root_path,'dynamo_project_b2/abp_align_eo/results/ite_0004/averages/temp.tbl')
    np.savetxt(table_temp_path,tomo_table,delimiter=' ',fmt='%s')
    subprocess.run(['dynamo2warp','-i',table_temp_path,'-tm',os.path.join(current_root_path,'dynamo/particles.reextract.doc'),
                    '-o', os.path.join(current_root_path,'temp.star')])
    
    warp_file = os.path.join(current_root_path,'temp.star')
    warp_helical = os.path.join(current_root_path,'particles_dynamo_b2.star')
    warp_rest_naming = os.path.join(current_root_path,'relion3_b2/particles.star')
    output_star = os.path.join(current_root_path,'particles_dynamo_b1.star')
    

    warp_header, warp_df = read_star(warp_file, header_lines=12)
    warp_3d_header, warp_3d_df = read_star(warp_rest_naming, header_lines=19)
    warp_helical_header, warp_helical_df = read_star(warp_helical, header_lines=17)

    warp_3d_df['particle_true_ID'] = warp_3d_df.iloc[:, 11].astype(str).str.split('_ali_').str[1].str.split('_ctf_').str[0].astype(int)
    particle_true_ids = warp_3d_df["particle_true_ID"].to_numpy()
    particle_mask = (tomo_table[:, 0].astype(int)-begin_number) - 1

    particle_mask_idx = particle_true_ids[particle_mask]
    


    warp_df.iloc[:,0:3] = warp_df.iloc[:,0:3].astype(float)*binning_factor
    warp_df.iloc[:,6] = warp_helical_df.iloc[0,6] # name
    warp_df['mag'] = warp_helical_df.iloc[0,7]
    warp_df['pixel'] = warp_helical_df.iloc[0,8]
    warp_df['helicalID'] = warp_helical_df.iloc[particle_mask,9].values
    warp_df['psi_flip'] = warp_helical_df.iloc[0,10] # psi flip
    warp_df['HelicalTrackLengthAngst'] = warp_helical_df.iloc[particle_mask,11].values
    
    new_header_lines = list(warp_header)  # Copy existing header lines
    new_header_lines.append("_rlnMagnification #8")
    new_header_lines.append("_rlnDetectorPixelSize #9")
    new_header_lines.append("_rlnHelicalTubeID #10")
    new_header_lines.append("_rlnAnglePsiFlipRatio #11")
    new_header_lines.append("_rlnHelicalTrackLengthAngst #12")
    
    # Write the new STAR file
    with open(output_star, 'w') as f:
        # Write header lines
        for line in new_header_lines:
            f.write(line.rstrip("\n") + "\n")
        # Write a blank line to separate header and data if required
    
        # Write dataframe to file using space as delimiter, without index and header
        warp_df.to_csv(f, sep=' ', index=False, header=False)
    
    begin_number = begin_number + np.max(tomo_table[:,0].astype(int))
    os.remove(warp_file)
    #os.remove(table_temp_path)



In [4]:
# Add angles, track length, prior angle, and random subset to the new star file


import pandas as pd
import numpy as np
import os
from scipy.spatial.transform import Rotation as R


# Function to read STAR file and separate header and data
def read_star(file_path, header_lines):
    """
    Reads a STAR file, skipping header_lines, returns header (as list of strings) and DataFrame.
    """
    # Read entire file as lines
    with open(file_path, 'r') as f:
        all_lines = f.readlines()
    
    # Separate header and data
    header = all_lines[:header_lines]
    # The remainder lines should be data rows. Assume whitespace-delimited.
    data_lines = all_lines[header_lines:]
    
    # Filter out any empty lines and comment lines starting with #
    data_lines = [line.strip() for line in data_lines if line.strip() and not line.startswith('#')]
    
    # Create a DataFrame by splitting each line on whitespace
    # Assuming all columns are numeric except perhaps the first one.
    data = [line.split() for line in data_lines]
    df = pd.DataFrame(data)
    return header, df


root_path = '/mnt/storage/data3/users/wen-lu/tau/warp/position_9/'
warp_file = os.path.join(root_path,'particles_dynamo_b1.star')
threeD_file = os.path.join(root_path,'relion/particles.star')
output_file = os.path.join(root_path,'relion/particles_new_relion5.star')
output_file_chimerax = os.path.join(root_path,'relion/particles_new_mod.star')


tomo_sizeX = 4096
tomo_sizeY = 4096
tomo_sizeZ = 3000
pixel_size = 2.42
binning = 4
inter_box_distance = 1



warp_header, warp_df = read_star(warp_file, header_lines=17)

# --- 2) Read 3D STAR and select matching rows ---
threeD_header, threeD_df = read_star(threeD_file, header_lines=45)


# assume column 1 is the “orig_filename” that embeds the particle number
threeD_df['relion_particlenumber'] = pd.to_numeric(threeD_df.iloc[:, 1], errors='coerce').astype('Int64')
idx = (threeD_df['relion_particlenumber'] - 1).to_numpy()

# # 1. READ ANGLES (From Relion 3D STAR)
# aligned_angles = threeD_df.iloc[:, 5:8].astype(float).to_numpy()

# # 2. DEFINE ROTATIONS (Using Lowercase 'zyz' for Intrinsic/Relion Standard)
# r_aligned = R.from_euler('ZYZ', aligned_angles, degrees=True)
# r_prior   = R.from_euler('ZYZ', [0.0, 90.0, 0.0], degrees=True)

# # 3. CALCULATE SUBTOMO ANGLES
# # Logic: The Final Alignment = Subtomo_Orientation * Prior_Offset
# # Therefore: Subtomo_Orientation = Final_Alignment * Inverse(Prior_Offset)
# r_subtomo = r_aligned * r_prior.inv()

# # 4. EXTRACT NEW ANGLES
# new_subtomo_angles = r_subtomo.as_euler('ZYZ', degrees=True)

# 5. UPDATE DATAFRAME
# threeD_df.iloc[:, 5:8] = new_subtomo_angles
threeD_df['HelicalTrackLengthAngst'] = warp_df.iloc[idx, 11].astype(str).values
threeD_df.drop(columns=['relion_particlenumber'], inplace=True)
threeD_df['rlnAnglePsiFlipRatio'] = 0.5
# threeD_df['rlnAngleRotPrior'] = threeD_df.iloc[:,5]
# threeD_df['rlnAngleTiltPrior'] = threeD_df.iloc[:,6]
# threeD_df['rlnAnglePsiPrior'] = threeD_df.iloc[:,7]
# threeD_df['rlnAngleRotPrior'] = 0
# threeD_df['rlnAngleTiltPrior'] = 90
# threeD_df['rlnAnglePsiPrior'] = 0
threeD_df['rlnHelicalTubeID'] = warp_df.iloc[idx, 9].astype(float).astype(int).values
threeD_df['rlnCenteredCoordinateXAngst'] = (threeD_df[2].astype(float)-tomo_sizeX/2)*pixel_size
threeD_df['rlnCenteredCoordinateYAngst'] = (threeD_df[3].astype(float)-tomo_sizeY/2)*pixel_size
threeD_df['rlnCenteredCoordinateZAngst'] = (threeD_df[4].astype(float)-tomo_sizeZ/2)*pixel_size
threeD_df['_rlnRandomSubset'] = 0
threeD_df['_rlnAngleRot'] = 0
threeD_df['_rlnAngleTilt'] = 0
threeD_df['_rlnAnglePsi'] = 0

# Assign random subset for each filament
filamentIDs = threeD_df['rlnHelicalTubeID'].unique()
for fid in filamentIDs:
    mask = threeD_df['rlnHelicalTubeID']== fid  
    # Put the same filament coordinarte into the same group
    threeD_df.loc[mask, '_rlnRandomSubset'] = float(fid) % 2 + 1

# drop helpers before writing
new_3D_df = threeD_df


# Update header to include the new column definitions.
# You need to find correct placement in header for new loop items.
# For simplicity, appending new definitions at the end of header.
new_header_lines = list(threeD_header)  # Copy existing header lines

# new_header_lines[35] = '_rlnTomoSubtomogramRot #6\n'
# new_header_lines[36] = '_rlnTomoSubtomogramTilt #7\n'
# new_header_lines[37] = '_rlnTomoSubtomogramPsi #8\n'

new_header_lines[35] = '_rlnAngleRotPrior #6\n'
new_header_lines[36] = '_rlnAngleTiltPrior #7\n'
new_header_lines[37] = '_rlnAnglePsiPrior #8\n'

new_header_lines.append("_rlnHelicalTrackLengthAngst #16")
new_header_lines.append("_rlnAnglePsiFlipRatio #17")
# new_header_lines.append("_rlnAngleRotPrior #18")
# new_header_lines.append("_rlnAngleTiltPrior #19")
# new_header_lines.append("_rlnAnglePsiPrior #20")
new_header_lines.append("_rlnHelicalTubeID #21")
new_header_lines.append("_rlnCenteredCoordinateXAngst #22")
new_header_lines.append("_rlnCenteredCoordinateYAngst #23")
new_header_lines.append("_rlnCenteredCoordinateZAngst #24")
new_header_lines.append("_rlnRandomSubset #25")
new_header_lines.append("_rlnAngleRot #26")
new_header_lines.append("_rlnAngleTilt #27")
new_header_lines.append("_rlnAnglePsi #28")



# Write the new STAR file
with open(output_file, 'w') as f:
    # Write header lines
    for line in new_header_lines:
        f.write(line.rstrip("\n") + "\n")
    # Write a blank line to separate header and data if required

    # Write dataframe to file using space as delimiter, without index and header
    new_3D_df.to_csv(f, sep=' ', index=False, header=False)

new_3D_df_chimerax = new_3D_df
new_3D_df_chimerax.iloc[:,0] = 10

# Write the new STAR file
with open(output_file_chimerax, 'w') as f:
    # Write header lines
    for line in new_header_lines:
        f.write(line.rstrip("\n") + "\n")
    # Write a blank line to separate header and data if required

    # Write dataframe to file using space as delimiter, without index and header
    new_3D_df_chimerax.to_csv(f, sep=' ', index=False, header=False)